# Supplementary Note 2 - The power system

Figures S17–S22: the TYNDP 2024 build-out the paper runs on, and how its reservoirs and batteries operate.

In [ ]:
import xarray as xr

from cordex2energy_analysis.config import (COMPARISON_SCENARIO, SUPPLY_SCENARIO, SUPPLY_YEAR,
                                           ZONE_LABELS, ZONE_ORDER, setup)
from cordex2energy_analysis.figures_system import (figure_s17, figure_s18, figure_s19,
                                                   operation_figure)
from cordex2energy_analysis.plotting_publication import publication_rc, save
from cordex2energy_analysis.power_system import (compare_build_outs, installed_capacities_by_country,
                                                 sum_by_zone)
from cordex2energy_analysis.runs import battery_throughput_curves, reservoir_operation_curves

context = setup("si_note2_system")

In [ ]:
supply_workbook = context.paths.tyndp / "20231103 - Final Supply Inputs for TYNDP 2024 Scenarios.xlsx"
capacities = {scenario: installed_capacities_by_country(supply_workbook, scenario, SUPPLY_YEAR)
              for scenario in (SUPPLY_SCENARIO, COMPARISON_SCENARIO)}
totals, added_by_zone, reference_by_zone = compare_build_outs(
    capacities, SUPPLY_SCENARIO, COMPARISON_SCENARIO, SUPPLY_YEAR, ZONE_ORDER)
baseload = xr.open_dataset(context.paths.baseload)["baseload"]
reservoir_countries, reservoir_storage, reservoir_dispatch = reservoir_operation_curves(context)
battery_throughput = battery_throughput_curves(context)

## S17 - The LOW/2050 build-out
**In:** TYNDP 2024 supply workbook, LOW/2050. **Shows:** installed generation and battery capacity per country (a, b) and per balancing zone (c, d).

In [ ]:
with publication_rc():
    save(figure_s17(capacities[SUPPLY_SCENARIO], sum_by_zone(capacities[SUPPLY_SCENARIO], ZONE_ORDER)),
         "S17_tyndp_low_capacities", context.figure_dir)

## S18 - HIGH/2050 against LOW/2050
**In:** the two TYNDP 2024 build-outs. **Shows:** Europe-wide capacity under each (a) and what HIGH adds per zone (b, c).

In [ ]:
with publication_rc():
    save(figure_s18(totals, added_by_zone, reference_by_zone), "S18_tyndp_low_vs_high",
         context.figure_dir)

## S19 - Baseload demand
**In:** the DE/2050 baseload typical year. **Shows:** weekly-mean baseload per balancing zone.

In [ ]:
with publication_rc():
    save(figure_s19(baseload), "S19_baseload_de_2050", context.figure_dir)

## S20 and S21 - Reservoir operation
**In:** the paper's run, per-country components. **Shows:** daily reservoir dispatch (S20) and storage (S21) over the year, per warming level.

In [ ]:
with publication_rc():
    save(operation_figure(reservoir_dispatch, context.style, mode="band", ncols=5,
                          ylabel="Reservoir dispatch (GWh d$^{-1}$)", height_mm=84),
         "S20_reservoir_dispatch", context.figure_dir)
    save(operation_figure(reservoir_storage, context.style, mode="spaghetti", ncols=5,
                          ylabel="Reservoir storage (GWh)", height_mm=84),
         "S21_reservoir_storage", context.figure_dir)

## S22 - Battery operation
**In:** the paper's run, zonal components. **Shows:** energy cycled through the batteries each day, per zone and warming level.

In [ ]:
with publication_rc():
    save(operation_figure({ZONE_LABELS[zone]: battery_throughput[zone] for zone in ZONE_ORDER},
                          context.style, mode="band", ncols=3, ylabel="GWh cycled d$^{-1}$",
                          height_mm=76),
         "S22_battery_throughput", context.figure_dir)